# UNIT V — NLP Practice Programs
**Aditya Kumar Singh | RA2311003010916 | Mrs. Sornalakshmi**

| # | Program | Library / Model | Technique |
|---|---------|----------------|-----------|
| 1 | Rule-Based Chatbot | NLTK (Chat utility) | Pattern Matching (Regex) |
| 2 | AI Chatbot (DialoGPT) | Transformers | Generative LM |
| 3 | Question Answering | Transformers (BERT) | Extractive QA |
| 4 | Extractive Summarization | NLTK | Word Frequency |
| 5 | Abstractive Summarization | Transformers (BART) | Seq2Seq |
| 6 | Machine Translation | Transformers (Opus-MT) | MarianMT |
| 7 | Machine Translation | googletrans | API-based |

## 📦 Installation
Run this cell once before executing any program.

In [1]:
!pip install nltk transformers torch sentencepiece googletrans==4.0.0-rc1 sacremoses -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastmcp 2.8.1 requires httpx>=0.28.1, but you have httpx 0.13.3 which is incompatible.
fastmcp 2.8.1 requires python-dotenv>=1.1.0, but you have python-dotenv 1.0.1 which is incompatible.
gradio 6.3.0 requires httpx<1.0,>=0.24.1, but you have httpx 0.13.3 which is incompatible.
gradio-client 2.0.3 requires httpx>=0.24.1, but you have httpx 0.13.3 which is incompatible.
langsmith 0.3.35 requires httpx<1,>=0.23.0, but you have httpx 0.13.3 which is incompatible.
mcp 1.9.4 requires httpx>=0.27, but you have httpx 0.13.3 which is incompatible.
openai 1.88.0 requires httpx<1,>=0.23.0, but you have httpx 0.13.3 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Program 1: Simple Rule-Based Chatbot using NLTK

**Aim:** Implement a simple rule-based chatbot using NLTK that matches patterns and generates predefined responses.

**Algorithm:**
1. Import NLTK's `chat` and `reflections` utilities.
2. Define a list of `(pattern, response-list)` pairs.
3. Create a `Chat` object with the patterns and reflections.
4. Call `converse()` to start an interactive session.
5. The bot replies until the user types `'quit'`.

> **Note:** The `Chat` class uses regex matching. `reflections` maps pronouns like `'I am'` to `'you are'` to make responses more natural.

In [ ]:
# Program 1: Simple Rule-Based Chatbot using NLTK
import nltk
from nltk.chat.util import Chat, reflections

nltk.download('punkt', quiet=True)

# Define conversation patterns
pairs = [
    (r"hi|hello|hey",
     ["Hello! How can I help you?",
      "Hi there! What can I do for you?"]),

    (r"what is your name?",
     ["I am Jason, your virtual assistant.",
      "My name is Jason!"]),

    (r"how are you?",
     ["I'm doing great, thank you!",
      "I'm fine. How about you?"]),

    (r"what can you do?",
     ["I can answer basic questions and have a conversation!"]),

    (r"tell me about (.*)",
     ["Sure! %1 is a very interesting topic.",
      "I know a bit about %1! What would you like to know?"]),

    (r"quit|bye|exit",
     ["Goodbye! Have a great day!",
      "Bye! Come back soon."]),

    (r"(.*)",
     ["I'm not sure I understand. Could you rephrase?",
      "Interesting! Tell me more."]),
]

def chatbot():
    print("Chatbot: Hello! I am Jason. Type 'quit' to exit.")
    chat = Chat(pairs, reflections)
    chat.converse()

if __name__ == "__main__":
    chatbot()

Chatbot: Hello! I am Jason. Type 'quit' to exit.
Hello! How can I help you?
My name is Jason!
My name is Jason!
I'm doing great, thank you!
Sure! (.*) is a very interesting topic.


---
## Program 2: AI Chatbot using Transformers (DialoGPT)

**Aim:** Build a generative AI chatbot using the pre-trained DialoGPT model that can maintain multi-turn conversations by encoding dialogue history.

**Algorithm:**
1. Load `microsoft/DialoGPT-medium` tokenizer and model.
2. Encode user input as token IDs and append EOS token.
3. Concatenate current input with past conversation history.
4. Call `model.generate()` to produce a response.
5. Decode output tokens back to text.
6. Repeat until user types `'quit'`.

> **Note:** DialoGPT is a generative model. Responses vary between runs due to sampling (top-k, top-p). Set `do_sample=False` for deterministic output.

In [ ]:
# Program 2: AI Chatbot using DialoGPT (Transformers)
import os
import warnings

# Silence non-fatal Windows / notebook environment warnings.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch


tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")

print("DialoGPT Chatbot started. Type 'quit' to exit.\n")

chat_history_ids = None  # stores conversation context

for step in range(6):  # allow 6 conversation turns
    user_input = input("You: ")
    if user_input.lower() in ["quit", "exit", "bye"]:
        print("Bot: Goodbye! Have a nice day!")
        break

    # Encode new user input + EOS token
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token, return_tensors="pt"
    )

    # Append to chat history
    bot_input_ids = (
        torch.cat([chat_history_ids, new_input_ids], dim=-1)
        if chat_history_ids is not None else new_input_ids
    )

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids, max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3, do_sample=True,
        top_k=50, top_p=0.92, temperature=0.75,
    )

    # Decode only the new response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )
    print(f"Bot: {response}")

DialoGPT Chatbot started. Type 'quit' to exit.



The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Bot: 
Bot: .
Bot: ..


---
## Program 3: Extractive Question Answering using BERT

**Aim:** Implement an Extractive QA system using a pre-trained BERT model that locates and extracts the answer to a question from a given context passage.

**Algorithm:**
1. Load pre-trained BERT QA tokenizer and model via the `pipeline` API.
2. Provide a context paragraph and a question.
3. The pipeline tokenizes both and runs inference.
4. BERT predicts start/end token positions of the answer span.
5. Decode and return the extracted answer text with confidence score.

In [2]:
# Program 3: Extractive Question Answering using BERT
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
import torch

# Load pre-trained BERT QA model and tokenizer directly
qa_tokenizer = AutoTokenizer.from_pretrained(
    "bert-large-uncased-whole-word-masking-finetuned-squad"
)
qa_model = AutoModelForQuestionAnswering.from_pretrained(
    "bert-large-uncased-whole-word-masking-finetuned-squad"
)

# -- Example 1: NLP Context -----------------------------------------------
context1 = """
Natural Language Processing (NLP) is a subfield of artificial intelligence
that focuses on the interaction between computers and human language.
NLP techniques are used in applications such as speech recognition, machine
translation, sentiment analysis, chatbots, and text summarization.
BERT (Bidirectional Encoder Representations from Transformers) is a
transformer-based model developed by Google in 2018. It is pre-trained on
large text corpora and can be fine-tuned for various NLP tasks including
question answering.
"""

questions1 = [
    "What is NLP?",
    "Who developed BERT?",
    "When was BERT developed?",
    "What tasks can BERT be fine-tuned for?",
]

print("=" * 60)
print(" EXAMPLE 1 -- Natural Language Processing")
print("=" * 60)
for q in questions1:
    inputs = qa_tokenizer(q, context1, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = qa_model(**inputs)
    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits) + 1
    answer_ids = inputs["input_ids"][0][start_idx:end_idx]
    result_answer = qa_tokenizer.decode(answer_ids, skip_special_tokens=True)
    print(f"Q: {q}")
    print(f"A: {result_answer}")
    print("-" * 40)

# -- Example 2: Machine Learning Context ----------------------------------
context2 = """
Machine Learning is a branch of artificial intelligence that enables computers
to learn from data and improve their performance without being explicitly
programmed. Supervised learning uses labeled data to train models, while
unsupervised learning discovers hidden patterns in unlabeled data.
Deep Learning is a subset of machine learning that uses neural networks with
many layers to learn complex patterns from large amounts of data.
"""

questions2 = [
    "What is machine learning?",
    "What does supervised learning use?",
    "What is deep learning?",
]

print("\n" + "=" * 60)
print(" EXAMPLE 2 -- Machine Learning")
print("=" * 60)
for q in questions2:
    inputs = qa_tokenizer(q, context2, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = qa_model(**inputs)
    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits) + 1
    answer_ids = inputs["input_ids"][0][start_idx:end_idx]
    result_answer = qa_tokenizer.decode(answer_ids, skip_special_tokens=True)
    print(f"Q: {q}")
    print(f"A: {result_answer}")
    print("-" * 40)

C:\Users\BITTU\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BITTU\.cache\huggingface\hub\models--bert-large-uncased-whole-word-masking-finetuned-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\BITTU\AppData\Roaming\Python\Python312\site-packages\transforme

 EXAMPLE 1 -- Natural Language Processing
Q: What is NLP?
A: 
----------------------------------------
Q: Who developed BERT?
A: google
----------------------------------------
Q: When was BERT developed?
A: 2018
----------------------------------------
Q: What tasks can BERT be fine-tuned for?
A: question answering
----------------------------------------

 EXAMPLE 2 -- Machine Learning
Q: What is machine learning?
A: a branch of artificial intelligence
----------------------------------------
Q: What does supervised learning use?
A: labeled data
----------------------------------------
Q: What is deep learning?
A: a subset of machine learning that uses neural networks with many layers to learn complex patterns from large amounts of data
----------------------------------------


---
## Program 4: Extractive Text Summarization (NLTK / Word Frequency)

**Aim:** Implement an extractive text summarization system using NLTK that scores sentences by word frequency and returns the top-N most informative sentences.

**Algorithm:**
1. Tokenize the input text into sentences and words.
2. Remove stopwords and punctuation.
3. Compute word frequency (normalized by max frequency).
4. Score each sentence by summing the frequencies of its words.
5. Select the top-N highest-scoring sentences (preserving original order).
6. Return the joined sentences as the summary.

> **Note:** Extractive summarization cannot rephrase. It works best when the input text has clear, self-contained sentences.

In [3]:
# Program 4: Extractive Text Summarization using NLTK
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
import heapq
import re

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

def extractive_summarize(text, num_sentences=3):
    # Step 1: Clean text
    clean_text = re.sub(r'\s+', ' ', text)
    clean_text = re.sub(r'[^a-zA-Z0-9. ]', '', clean_text)

    # Step 2: Tokenize
    sentences = sent_tokenize(text)
    words = word_tokenize(clean_text.lower())
    stop_words = set(stopwords.words('english'))
    filtered = [w for w in words if w not in stop_words and w.isalnum()]

    # Step 3: Word frequency (normalized)
    freq = {}
    for w in filtered:
        freq[w] = freq.get(w, 0) + 1
    max_freq = max(freq.values()) if freq else 1
    freq = {w: f / max_freq for w, f in freq.items()}

    # Step 4: Score sentences
    scores = {}
    for sent in sentences:
        for w in word_tokenize(sent.lower()):
            if w in freq:
                scores[sent] = scores.get(sent, 0) + freq[w]

    # Step 5: Select top-N (preserve order)
    best = heapq.nlargest(num_sentences, scores, key=scores.get)
    summary = ' '.join([s for s in sentences if s in best])
    return summary


text = """
Artificial Intelligence (AI) is transforming the modern world in unprecedented ways.
Machine Learning, a subset of AI, allows computers to learn from large amounts of
data without being explicitly programmed. Deep Learning, using multi-layered neural
networks, has achieved remarkable results in image recognition, natural language
processing, and speech recognition. Natural Language Processing (NLP) is a critical
branch of AI that enables machines to understand, interpret, and generate human
language. Applications of NLP include machine translation, sentiment analysis,
chatbots, and text summarization. Text summarization helps users quickly grasp the
key points of a document without reading the entire content. Extractive summarization
selects important sentences directly from the text, while abstractive summarization
generates new sentences that capture the core ideas.
"""

print("EXTRACTIVE SUMMARY (3 sentences)")
print("=" * 60)
print(extractive_summarize(text, num_sentences=3))

print("\nEXTRACTIVE SUMMARY (2 sentences)")
print("=" * 60)
print(extractive_summarize(text, num_sentences=2))

EXTRACTIVE SUMMARY (3 sentences)
Deep Learning, using multi-layered neural
networks, has achieved remarkable results in image recognition, natural language
processing, and speech recognition. Natural Language Processing (NLP) is a critical
branch of AI that enables machines to understand, interpret, and generate human
language. Extractive summarization
selects important sentences directly from the text, while abstractive summarization
generates new sentences that capture the core ideas.

EXTRACTIVE SUMMARY (2 sentences)
Natural Language Processing (NLP) is a critical
branch of AI that enables machines to understand, interpret, and generate human
language. Extractive summarization
selects important sentences directly from the text, while abstractive summarization
generates new sentences that capture the core ideas.


---
## Program 5: Abstractive Text Summarization (Transformers / BART)

**Aim:** Implement abstractive text summarization using a pre-trained BART model via Hugging Face transformers that generates concise, human-like summaries.

**Algorithm:**
1. Load the `facebook/bart-large-cnn` summarization pipeline.
2. Pass the input text along with `max_length` and `min_length` parameters.
3. The model encodes the input and auto-regressively decodes a summary.
4. Return and display the generated summary text.

> **Note:** BART generates summaries that may not appear verbatim in the source text. Adjust `max_length` and `min_length` to control summary length.

In [4]:
# Program 5: Abstractive Summarization using BART (Transformers)
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load pre-trained BART summarization model and tokenizer directly
bart_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

# -- Example 1: COVID-19 -----------------------------------------------
text1 = """
The COVID-19 pandemic has caused unprecedented global disruption since its
emergence in late 2019. The virus, SARS-CoV-2, spread rapidly across the world,
leading to millions of deaths and widespread economic turmoil. Governments
worldwide implemented lockdowns, travel restrictions, and social distancing
measures to slow the virus's spread. Healthcare systems were stretched to their
limits as hospitals struggled to cope with the surge in patients. The pandemic
accelerated the development and rollout of vaccines at a record pace, with
multiple vaccines receiving emergency authorization within a year. Despite the
vaccines, new variants continued to emerge, posing challenges to global health.
The pandemic also highlighted existing health disparities and the importance of
global cooperation in addressing public health crises.
"""

print("EXAMPLE 1: COVID-19")
print("=" * 60)
print("Original Text:", text1[:200], "...")
inputs1 = bart_tokenizer(text1, return_tensors="pt", truncation=True, max_length=1024)
summary_ids1 = bart_model.generate(
    inputs1["input_ids"],
    attention_mask=inputs1.get("attention_mask"),
    max_length=80,
    min_length=30,
    do_sample=False,
)
summary1 = bart_tokenizer.decode(summary_ids1[0], skip_special_tokens=True)
print("\nAbstractive Summary:")
print(summary1)

# -- Example 2: Climate Change -----------------------------------------
text2 = """
Climate change refers to long-term shifts in global temperatures and weather
patterns. While some natural processes cause climate change, human activities
have been the main driver since the 1800s. The burning of fossil fuels such as
coal, oil, and gas generates greenhouse gas emissions that trap the sun's heat
and raise temperatures. Rising temperatures lead to more frequent and severe
weather events like hurricanes, floods, and droughts. Sea levels are rising
due to melting polar ice caps, threatening coastal communities. Scientists and
environmentalists are calling for urgent action to reduce carbon emissions and
transition to renewable energy sources to mitigate the worst effects.
"""

print("\nEXAMPLE 2: Climate Change")
print("=" * 60)
inputs2 = bart_tokenizer(text2, return_tensors="pt", truncation=True, max_length=1024)
summary_ids2 = bart_model.generate(
    inputs2["input_ids"],
    attention_mask=inputs2.get("attention_mask"),
    max_length=70,
    min_length=25,
    do_sample=False,
)
summary2 = bart_tokenizer.decode(summary_ids2[0], skip_special_tokens=True)
print("Abstractive Summary:")
print(summary2)

C:\Users\BITTU\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BITTU\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\BITTU\AppData\Roaming\Python\Python312\site-packages\transformers\tokenization_utils_base.py

EXAMPLE 1: COVID-19
Original Text: 
The COVID-19 pandemic has caused unprecedented global disruption since its
emergence in late 2019. The virus, SARS-CoV-2, spread rapidly across the world,
leading to millions of deaths and widespread ...

Abstractive Summary:
The COVID-19 pandemic has caused unprecedented global disruption since its emergence in late 2019. The virus, SARS-CoV-2, spread rapidly across the world, leading to millions of deaths and widespread economic turmoil. The pandemic also highlighted existing health disparities and the importance of global cooperation in addressing public health crises.

EXAMPLE 2: Climate Change
Abstractive Summary:
Climate change refers to long-term shifts in global temperatures and weather patterns. Human activities have been the main driver since the 1800s. Rising temperatures lead to more frequent and severeweather events like hurricanes, floods, and droughts.


---
## Program 6: Machine Translation using Transformers (Helsinki-NLP / Opus-MT)

**Aim:** Implement machine translation using the pre-trained Helsinki-NLP Opus-MT model that translates text between multiple language pairs.

**Algorithm:**
1. Load MarianMT tokenizer and model for a specific language pair (e.g., `Helsinki-NLP/opus-mt-en-fr`).
2. Tokenize the source text using `MarianTokenizer`.
3. Call `model.generate()` to produce translated token IDs.
4. Decode the token IDs back to target language text.
5. Repeat for different language pairs.

> **Note:** Model names follow the pattern `Helsinki-NLP/opus-mt-{src}-{tgt}`. Available pairs include `en-fr`, `en-de`, `en-es`, `en-hi`, `fr-en`, `de-en`, etc.

In [5]:
# Program 6: Machine Translation using Helsinki-NLP Opus-MT
from transformers import MarianMTModel, MarianTokenizer

def translate(text, src_lang, tgt_lang):
    model_name = f"Helsinki-NLP/opus-mt-{src_lang}-{tgt_lang}"
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name)
    tokens = tokenizer([text], return_tensors="pt", padding=True)
    translated = model.generate(**tokens, num_beams=4, early_stopping=True)
    return tokenizer.decode(translated[0], skip_special_tokens=True)


sentences = [
    "Natural Language Processing is a fascinating field of AI.",
    "Machine translation helps bridge the communication gap.",
    "Deep learning models can understand and generate human language.",
]

# -- English to French ---------------------------------------------------
print("English -> French")
print("=" * 60)
for s in sentences:
    print(f"EN: {s}")
    print(f"FR: {translate(s, 'en', 'fr')}")
    print("-" * 50)

# -- English to German ---------------------------------------------------
print("\nEnglish -> German")
print("=" * 60)
for s in sentences:
    print(f"EN: {s}")
    print(f"DE: {translate(s, 'en', 'de')}")
    print("-" * 50)

# -- English to Spanish -------------------------------------------------
print("\nEnglish -> Spanish")
print("=" * 60)
for s in sentences:
    print(f"EN: {s}")
    print(f"ES: {translate(s, 'en', 'es')}")
    print("-" * 50)

English -> French
EN: Natural Language Processing is a fascinating field of AI.


C:\Users\BITTU\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BITTU\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-fr. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


FR: Le traitement du langage naturel est un domaine fascinant de l'IA.
--------------------------------------------------
EN: Machine translation helps bridge the communication gap.
FR: La traduction automatique aide à combler l'écart de communication.
--------------------------------------------------
EN: Deep learning models can understand and generate human language.
FR: Les modèles d'apprentissage profond peuvent comprendre et générer le langage humain.
--------------------------------------------------

English -> German
EN: Natural Language Processing is a fascinating field of AI.


C:\Users\BITTU\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BITTU\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-de. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


DE: Natürliche Sprachverarbeitung ist ein faszinierendes Feld der KI.
--------------------------------------------------
EN: Machine translation helps bridge the communication gap.
DE: Maschinelle Übersetzung hilft, die Kommunikationslücke zu überbrücken.
--------------------------------------------------
EN: Deep learning models can understand and generate human language.
DE: Deep-Learning-Modelle können menschliche Sprache verstehen und erzeugen.
--------------------------------------------------

English -> Spanish
EN: Natural Language Processing is a fascinating field of AI.


C:\Users\BITTU\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BITTU\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-es. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


ES: El procesamiento del lenguaje natural es un campo fascinante de la IA.
--------------------------------------------------
EN: Machine translation helps bridge the communication gap.
ES: Traducción automática ayuda a cerrar la brecha de comunicación.
--------------------------------------------------
EN: Deep learning models can understand and generate human language.
ES: Los modelos de aprendizaje profundo pueden entender y generar lenguaje humano.
--------------------------------------------------


---
## Program 7: Machine Translation using Google Translate API (googletrans)

**Aim:** Demonstrate machine translation using the `googletrans` library, including automatic source language detection and batch translation between multiple language pairs.

**Algorithm:**
1. Install and import the `googletrans` library.
2. Create a `Translator` object.
3. Call `translator.translate(text, dest='target_lang')`.
4. Access `result.text` for the translated output and `result.src` for the detected source language.
5. Print source language, target language, and translated text.

> **Note:** `googletrans` uses Google's web API (no official key needed for small usage). For production, consider the official Google Cloud Translation API.

In [6]:
# Program 7: Machine Translation using googletrans
# Install: pip install googletrans==4.0.0-rc1
from googletrans import Translator, LANGUAGES

translator = Translator()

# -- Part A: Multi-Language Translation ----------------------------------
sentence = "Artificial Intelligence is changing the world."
target_langs = {
    'fr': 'French',
    'de': 'German',
    'es': 'Spanish',
    'hi': 'Hindi',
    'ta': 'Tamil',
    'ja': 'Japanese',
    'zh-cn': 'Chinese (Simplified)',
    'ar': 'Arabic',
}

print("PART A: Multi-Language Translation")
print("=" * 60)
print(f"Source (English): {sentence}\n")
for lang_code, lang_name in target_langs.items():
    result = translator.translate(sentence, dest=lang_code)
    print(f"{lang_name:25s}: {result.text}")

# -- Part B: Auto Language Detection + Translation to English -----------
print("\nPART B: Auto Language Detection + Translate to English")
print("=" * 60)
foreign_sentences = [
    "Bonjour, comment allez-vous?",       # French
    "Guten Morgen, wie geht es Ihnen?",   # German
    "Hola, como estas?",                  # Spanish
    "Namaste, aap kaise hain?",           # Hindi
    "Vanakkam, eppadi irukkireekal?",     # Tamil
]

for sent in foreign_sentences:
    detected = translator.detect(sent)
    translated = translator.translate(sent, dest='en')
    lang_name = LANGUAGES.get(detected.lang, detected.lang).capitalize()
    print(f"Input    : {sent}")
    print(f"Detected : {lang_name} (confidence: {detected.confidence:.2f})")
    print(f"English  : {translated.text}")
    print("-" * 50)

PART A: Multi-Language Translation
Source (English): Artificial Intelligence is changing the world.

French                   : L'intelligence artificielle change le monde.
German                   : Künstliche Intelligenz verändert die Welt.
Spanish                  : La Inteligencia Artificial está cambiando el mundo.
Hindi                    : आर्टिफिशियल इंटेलिजेंस दुनिया को बदल रहा है।
Tamil                    : செயற்கை நுண்ணறிவு உலகை மாற்றுகிறது.
Japanese                 : 人工知能は世界を変えています。
Chinese (Simplified)     : 人工智能正在改变世界。
Arabic                   : الذكاء الاصطناعي يغير العالم.

PART B: Auto Language Detection + Translate to English
Input    : Bonjour, comment allez-vous?


TypeError: unsupported format string passed to NoneType.__format__

---
## Summary — All Programs

| # | Program | Library / Model | Technique |
|---|---------|----------------|-----------|
| 1 | Rule-Based Chatbot | NLTK (Chat utility) | Pattern Matching (Regex) |
| 2 | AI Chatbot | Transformers (DialoGPT) | Generative Language Model |
| 3 | Question Answering | Transformers (BERT-SQuAD) | Extractive QA (Span Prediction) |
| 4 | Extractive Summarization | NLTK | Word Frequency Scoring |
| 5 | Abstractive Summarization | Transformers (BART-CNN) | Seq2Seq / Encoder-Decoder |
| 6 | Machine Translation | Transformers (Opus-MT) | MarianMT Neural Translation |
| 7 | Machine Translation | googletrans (Google API) | API-based Neural Translation |

### Required Installations
```bash
pip install nltk transformers torch sentencepiece googletrans==4.0.0-rc1 sacremoses
```

### NLTK Data Downloads
```python
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
```